In [1]:
# WVS Data Preprocessing with Ollama
# This notebook preprocesses the World Values Survey trend file by:
# 1. Loading the dataset
# 2. Decoding encrypted column names using documentation
# 3. Using Ollama for intelligent preprocessing
# 4. Displaying the first 10 rows of processed data

In [5]:
# Cell 1: Import Required Libraries

import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import re
import json
import time
from pathlib import Path
import warnings
import os
warnings.filterwarnings('ignore')

# For Ollama integration
import subprocess
import sys

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [6]:
# Cell 2: Setup and Diagnose Ollama Connection

def diagnose_ollama():
    """Diagnose Ollama installation and connection"""
    
    print("🔍 DIAGNOSING OLLAMA CONNECTION")
    print("="*60)
    
    # Check if ollama is in PATH
    import shutil
    ollama_path = shutil.which('ollama')
    if ollama_path:
        print(f"✅ Ollama executable found: {ollama_path}")
    else:
        print("❌ Ollama not found in PATH")
        print("   Please install Ollama from: https://ollama.com/download")
        return False
    
    # Check version
    try:
        result = subprocess.run(['ollama', '--version'], 
                              capture_output=True, text=True, timeout=5)
        print(f"✅ Ollama version: {result.stdout.strip()}")
    except:
        print("⚠️ Could not determine version")
    
    # Check if service is running via API
    try:
        response = requests.get('http://localhost:11434/api/tags', timeout=3)
        if response.status_code == 200:
            print("✅ Ollama service is running (API accessible)")
            models = response.json().get('models', [])
            if models:
                print(f"   Available models: {[m['name'] for m in models]}")
            else:
                print("   No models found. Pull a model with: ollama pull llama2")
            return True
        else:
            print(f"❌ Ollama API returned status {response.status_code}")
    except requests.exceptions.ConnectionError:
        print("❌ Cannot connect to Ollama API")
        print("\n📢 TO START OLLAMA:")
        print("   1. Press Win + R, type 'ollama' and press Enter")
        print("   2. Or run 'ollama serve' in command prompt")
        print("   3. After starting, pull a model: ollama pull llama2")
    except Exception as e:
        print(f"❌ Error: {e}")
    
    return False

def query_ollama(prompt, model="llama2"):
    """Query Ollama with proper error handling"""
    
    # Try API first (most reliable)
    try:
        response = requests.post(
            'http://localhost:11434/api/generate',
            json={'model': model, 'prompt': prompt, 'stream': False},
            timeout=30
        )
        if response.status_code == 200:
            return response.json().get('response', '')
    except:
        pass
    
    # Try command line as fallback
    try:
        # Use shell=True for Windows
        result = subprocess.run(
            f'ollama run {model} "{prompt}"',
            shell=True,
            capture_output=True,
            text=True,
            encoding='utf-8',
            errors='ignore',
            timeout=45
        )
        if result.returncode == 0 and result.stdout:
            return result.stdout.strip()
    except:
        pass
    
    return None

# Run diagnosis
ollama_available = diagnose_ollama()

🔍 DIAGNOSING OLLAMA CONNECTION
✅ Ollama executable found: C:\Users\HP\AppData\Local\Programs\Ollama\ollama.EXE
✅ Ollama version: ollama version is 0.16.2
✅ Ollama service is running (API accessible)
   Available models: ['llama2:latest']


In [7]:
# Cell 3: Load the Dataset

def load_wvs_data(file_path):
    """
    Load WVS trend file with proper handling
    """
    try:
        if not os.path.exists(file_path):
            print(f"❌ File not found: {file_path}")
            return None
            
        print(f"Loading file: {file_path}")
        df = pd.read_stata(file_path, convert_categoricals=False)
        
        print(f"✅ Dataset loaded successfully")
        print(f"   Shape: {df.shape}")
        print(f"   Rows: {len(df)}")
        print(f"   Columns: {len(df.columns)}")
        
        return df
        
    except Exception as e:
        print(f"❌ Error loading dataset: {e}")
        return None

# Specify your file path
file_path = r'C:\Users\HP\Documents\GitHub\DWaV-project\Trends_VS_1981_2022_Stata_v4_1.dta'

# Load the data
df = load_wvs_data(file_path)

# If loading fails, create sample data
if df is None:
    print("\n⚠️ Creating sample WVS data for demonstration...")
    np.random.seed(42)
    n_rows = 100
    
    sample_data = {
        'studyno': np.random.randint(1, 100, n_rows),
        'version': ['v4.1.0'] * n_rows,
        'doi': ['10.14281/18241.27'] * n_rows,
        'S002': np.random.choice([840, 826, 276, 250, 643], n_rows),
        'S003': np.random.choice([840, 826, 276, 250, 643], n_rows),
        'S009': np.random.choice([1990, 1995, 2000, 2005, 2010, 2015, 2020], n_rows),
        'S020': np.random.choice([1, 2, 3, 4, 5, 6, 7], n_rows),
        'X001': np.random.choice([1, 2], n_rows),
        'X003': np.random.randint(18, 90, n_rows),
        'X003R': np.random.choice([1, 2, 3, 4, 5, 6], n_rows),
        'X007': np.random.choice([1, 2, 3, 4, 5, 6], n_rows),
        'X011': np.random.randint(0, 10, n_rows),
        'X025': np.random.choice([1, 2, 3, 4, 5, 6, 7, 8], n_rows),
        'A001': np.random.choice([1, 2, 3, 4], n_rows),
        'A002': np.random.choice([1, 2, 3, 4], n_rows),
        'A003': np.random.choice([1, 2, 3, 4], n_rows),
        'A004': np.random.choice([1, 2, 3, 4], n_rows),
        'A005': np.random.choice([1, 2, 3, 4], n_rows),
        'A006': np.random.choice([1, 2, 3, 4], n_rows),
        'A008': np.random.choice([1, 2, 3, 4], n_rows),
        'A009': np.random.choice([1, 2, 3, 4, 5], n_rows),
        'A170': np.random.randint(1, 11, n_rows),
        'E069': np.random.choice([1, 2, 3, 4], n_rows),
        'E069_01': np.random.choice([1, 2, 3, 4], n_rows),
        'E179WVS': np.random.choice([1, 2, 3, 4, 5], n_rows),
        'F114': np.random.randint(1, 11, n_rows),
        'G006': np.random.choice([1, 2, 3, 4], n_rows),
    }
    
    df = pd.DataFrame(sample_data)
    print(f"✅ Sample data created with shape: {df.shape}")
    print(f"   Columns: {list(df.columns)}")

Loading file: C:\Users\HP\Documents\GitHub\DWaV-project\Trends_VS_1981_2022_Stata_v4_1.dta
✅ Dataset loaded successfully
   Shape: (442473, 732)
   Rows: 442473
   Columns: 732


In [8]:
# Cell 4: Analyze Dataset Structure

print("📊 DATASET ANALYSIS")
print("="*60)

print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

# Group columns by prefix
prefixes = {}
for col in df.columns:
    prefix = col.split('_')[0][:1]
    prefixes[prefix] = prefixes.get(prefix, 0) + 1

print(f"\nColumn groups:")
for prefix, count in sorted(prefixes.items()):
    print(f"  {prefix}: {count} columns")

# Get all column names
print(f"\nAll columns ({len(df.columns)} total):")
for i, col in enumerate(df.columns[:15]):
    print(f"  {i+1:2d}. {col}")
if len(df.columns) > 15:
    print(f"  ... and {len(df.columns) - 15} more")

# Check data types
print(f"\nData types:")
print(df.dtypes.value_counts())

📊 DATASET ANALYSIS
Total rows: 442473
Total columns: 732

Column groups:
  A: 159 columns
  B: 8 columns
  C: 48 columns
  D: 51 columns
  E: 176 columns
  F: 89 columns
  G: 42 columns
  H: 23 columns
  I: 2 columns
  S: 27 columns
  V: 11 columns
  W: 4 columns
  X: 46 columns
  Y: 37 columns
  d: 1 columns
  m: 1 columns
  p: 1 columns
  s: 3 columns
  v: 2 columns
  x: 1 columns

All columns (732 total):
   1. studyno
   2. version
   3. doi
   4. stdyno_w
   5. versn_w
   6. S001
   7. s002
   8. S002VS
   9. S003
  10. COUNTRY_ALPHA
  11. COW_NUM
  12. COW_ALPHA
  13. S004
  14. S006
  15. S007
  ... and 717 more

Data types:
int8       640
float64     40
int32       20
int16       17
object      12
float32      3
Name: count, dtype: int64


In [ ]:
# Cell 5: Decode Columns with Ollama

def decode_with_ollama(column_name):
    """
    Use Ollama to decode what a WVS column means
    Returns a dictionary with name, description, and scale
    """
    
    prompt = f"""You are an expert on the World Values Survey (WVS). 
The column code '{column_name}' appears in the WVS dataset.

Provide information in this exact JSON format:
{{
    "name": "short variable name",
    "description": "brief description of what it measures",
    "scale": "response scale or format"
}}

If unsure, make your best guess based on WVS naming conventions."""
    
    response = query_ollama(prompt)
    
    if response:
        try:
            # Try to extract JSON from response
            import json
            import re
            
            # Find JSON in response
            json_match = re.search(r'\{.*\}', response, re.DOTALL)
            if json_match:
                return json.loads(json_match.group())
        except:
            pass
        
        # If JSON parsing fails, return structured response
        return {
            "name": column_name,
            "description": response[:100] + "...",
            "scale": "Unknown"
        }
    
    # Fallback for common columns if Ollama fails
    fallback = {
        'studyno': {"name": "Study Number", "description": "Unique identifier for the study/wave", "scale": "Numeric"},
        'version': {"name": "Dataset Version", "description": "Version of the dataset", "scale": "String"},
        'doi': {"name": "Digital Object Identifier", "description": "DOI for citing the dataset", "scale": "String"},
        'S002': {"name": "Country Code", "description": "Country identifier for the survey", "scale": "Numeric code"},
        'S003': {"name": "ISO Country Code", "description": "Numeric ISO country code", "scale": "3-digit numeric"},
        'S009': {"name": "Year", "description": "Year of survey", "scale": "4-digit year"},
        'S020': {"name": "Wave", "description": "Survey wave number", "scale": "1-7"},
        'X001': {"name": "Gender", "description": "Respondent's gender", "scale": "1=Male, 2=Female"},
        'X003': {"name": "Age", "description": "Respondent's age", "scale": "Years"},
        'A001': {"name": "Importance of Family", "description": "How important is family in life", "scale": "1=Very to 4=Not at all"},
        'A002': {"name": "Importance of Friends", "description": "How important are friends", "scale": "1=Very to 4=Not at all"},
        'E069': {"name": "Confidence in Churches", "description": "Confidence in religious institutions", "scale": "1=Great deal to 4=None"},
        'F114': {"name": "Life Satisfaction", "description": "Satisfaction with life", "scale": "1-10"},
        'G006': {"name": "National Pride", "description": "How proud of nationality", "scale": "1=Very to 4=Not proud"},
    }
    
    return fallback.get(column_name, {
        "name": column_name,
        "description": "Unknown WVS variable",
        "scale": "Unknown"
    })

def decode_all_columns(df, max_columns=10):
    """
    Decode all columns or first max_columns
    """
    
    if not ollama_available:
        print("⚠️ Ollama not available, using fallback knowledge base")
    
    columns_to_decode = df.columns[:max_columns]
    print(f"\n🔄 Decoding {len(columns_to_decode)} columns with Ollama...")
    
    decoded = {}
    for i, col in enumerate(columns_to_decode, 1):
        print(f"  {i}/{len(columns_to_decode)}: {col}")
        
        if ollama_available:
            result = decode_with_ollama(col)
            decoded[col] = result
            print(f"    → {result.get('name', col)}")
            time.sleep(1)  # Be nice to Ollama
        else:
            # Use fallback
            decoded[col] = decode_with_ollama(col)  # This will use fallback
        
    return decoded

# Decode columns
column_info = decode_all_columns(df, max_columns=15)

print(f"\n✅ Decoded {len(column_info)} columns")


🔄 Decoding 15 columns with Ollama...
  1/15: studyno
    → Study ID
  2/15: version
    → version
  3/15: doi
    → Doctrinal Involvement
  4/15: stdyno_w
    → Standardized world values score
  5/15: versn_w
    → Versn_W
  6/15: S001
    → SocialDistance
  7/15: s002
    → Religiosity
  8/15: S002VS
    → Political Preferences
  9/15: S003
    → S003
  10/15: COUNTRY_ALPHA
    → Country Alpha
  11/15: COW_NUM
    → Cow Number
  12/15: COW_ALPHA


In [ ]:
# Cell 6: Create Column Mapping

def create_column_mapping(column_info):
    """
    Create readable column names from decoded information
    """
    mapping = {}
    
    for orig, info in column_info.items():
        # Create a clean column name
        name = info.get('name', orig)
        # Remove special characters, replace spaces with underscores
        clean_name = re.sub(r'[^\w\s-]', '', name)
        clean_name = clean_name.replace(' ', '_').lower()
        # Limit length
        if len(clean_name) > 40:
            clean_name = clean_name[:40]
        
        mapping[orig] = {
            'new_name': clean_name,
            'description': info.get('description', ''),
            'scale': info.get('scale', '')
        }
    
    return mapping

import re
column_mapping = create_column_mapping(column_info)

print("📋 Column mapping created:")
for orig, info in list(column_mapping.items())[:10]:
    print(f"\n  {orig}")
    print(f"  → {info['new_name']}")
    print(f"    {info['description']}")

In [ ]:
# Cell 7: Get Preprocessing Suggestions

def get_preprocessing_suggestions(df, column_info):
    """
    Get preprocessing suggestions from Ollama
    """
    
    if not ollama_available:
        print("⚠️ Ollama not available, using default suggestions")
        return [
            "Handle missing values (replace -1 to -5 with NaN)",
            "Convert categorical variables to proper types",
            "Create derived variables for analysis",
            "Check for outliers in numeric columns",
            "Document all preprocessing steps"
        ]
    
    # Prepare context
    context = f"""
Dataset: World Values Survey
Rows: {len(df)}
Columns: {len(df.columns)}

Sample columns and their meanings:
"""
    for orig, info in list(column_info.items())[:5]:
        context += f"- {orig}: {info.get('description', 'Unknown')}\n"
    
    prompt = context + """
Based on this WVS data structure, suggest 5 specific preprocessing steps.
Focus on data cleaning, handling missing values, and preparing for analysis.
List each step briefly."""
    
    print("🤖 Getting preprocessing suggestions from Ollama...")
    response = query_ollama(prompt)
    
    if response:
        print("\n📋 Ollama's suggestions:")
        print("-"*40)
        print(response)
        return response.split('\n')
    else:
        return ["Could not get suggestions from Ollama"]

preprocessing_suggestions = get_preprocessing_suggestions(df, column_info)

In [ ]:
# Cell 8: Apply Preprocessing

def preprocess_wvs_data(df, column_mapping):
    """
    Apply preprocessing steps to the dataset
    """
    print("\n🔄 Starting preprocessing...")
    df_processed = df.copy()
    
    # Step 1: Rename columns
    rename_dict = {orig: info['new_name'] for orig, info in column_mapping.items()}
    df_processed = df_processed.rename(columns=rename_dict)
    print(f"   ✅ Renamed {len(rename_dict)} columns")
    
    # Step 2: Handle missing values
    missing_codes = [-5, -4, -3, -2, -1]
    
    # Try to convert to numeric where possible
    for col in df_processed.columns:
        try:
            df_processed[col] = pd.to_numeric(df_processed[col], errors='ignore')
        except:
            pass
    
    # Replace missing codes with NaN
    df_processed = df_processed.replace(missing_codes, np.nan)
    print(f"   ✅ Replaced missing codes with NaN")
    
    # Step 3: Basic data type optimization
    for col in df_processed.select_dtypes(include=['float64', 'int64']).columns:
        if df_processed[col].dropna().apply(float.is_integer).all():
            df_processed[col] = pd.to_numeric(df_processed[col], downcast='integer')
        else:
            df_processed[col] = pd.to_numeric(df_processed[col], downcast='float')
    
    print(f"   ✅ Optimized data types")
    
    # Step 4: Add metadata
    df_processed['_preprocessed_date'] = pd.Timestamp.now().date()
    df_processed['_preprocessing_method'] = 'Ollama-enhanced'
    df_processed['_source'] = 'WVS Trend File'
    
    print(f"\n✅ Preprocessing complete")
    print(f"   Final shape: {df_processed.shape}")
    print(f"   Memory: {df_processed.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    return df_processed

# Apply preprocessing
df_processed = preprocess_wvs_data(df, column_mapping)

In [ ]:
# Cell 9: Display Results

print("\n" + "="*80)
print("FIRST 10 ROWS OF PREPROCESSED WVS DATA")
print("="*80)

# Configure display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 40)

# Display first 10 rows
display(df_processed.head(10))

# Show transformation
print("\n" + "="*80)
print("COLUMN TRANSFORMATIONS")
print("="*80)

transform_data = []
for orig, new in zip(list(df.columns)[:10], list(df_processed.columns)[:10]):
    transform_data.append({
        'Original': orig,
        'New Name': new,
        'Description': column_mapping.get(orig, {}).get('description', '')[:50]
    })

transform_df = pd.DataFrame(transform_data)
display(transform_df)

In [ ]:
# Cell 10: Data Quality Report

print("📊 DATA QUALITY REPORT")
print("="*60)

# Basic metrics
total_cells = df_processed.shape[0] * df_processed.shape[1]
missing_cells = df_processed.isna().sum().sum()
missing_pct = (missing_cells / total_cells) * 100

print(f"Total cells: {total_cells:,}")
print(f"Missing cells: {missing_cells:,} ({missing_pct:.2f}%)")

# Columns with most missing values
missing_by_col = df_processed.isna().sum().sort_values(ascending=False)
print("\nColumns with most missing values:")
for col, missing in missing_by_col.head(5).items():
    if missing > 0:
        pct = (missing / len(df_processed)) * 100
        print(f"  {col}: {missing} missing ({pct:.1f}%)")

# Data types summary
print("\nData types:")
for dtype in df_processed.dtypes.value_counts().items():
    print(f"  {dtype[0]}: {dtype[1]} columns")

# Memory usage
memory_mb = df_processed.memory_usage(deep=True).sum() / 1024**2
print(f"\nMemory usage: {memory_mb:.2f} MB")

In [ ]:
# Cell 11: Summary of Ollama's Work

print("🤖 OLLAMA CONTRIBUTION SUMMARY")
print("="*60)

print(f"Columns analyzed by Ollama: {len(column_info)}")
print(f"Unique descriptions generated: {len(set([info.get('description', '') for info in column_info.values()]))}")
print(f"Preprocessing suggestions: {len(preprocessing_suggestions) if preprocessing_suggestions else 0}")

print("\nExample of Ollama's work:")
if column_info:
    first_col = list(column_info.keys())[0]
    print(f"\nColumn: {first_col}")
    print(f"Ollama said:")
    print(f"  Name: {column_info[first_col].get('name', 'N/A')}")
    print(f"  Description: {column_info[first_col].get('description', 'N/A')}")
    print(f"  Scale: {column_info[first_col].get('scale', 'N/A')}")

In [ ]:
# Cell 12: Save Results

def save_results(df_processed, column_info, column_mapping):
    """
    Save processed data and metadata
    """
    
    # Create output directory
    output_dir = 'wvs_processed'
    os.makedirs(output_dir, exist_ok=True)
    
    # Save processed data
    data_file = os.path.join(output_dir, 'wvs_processed.parquet')
    df_processed.to_parquet(data_file, index=False)
    print(f"✅ Data saved to {data_file}")
    
    # Save column mapping
    mapping_file = os.path.join(output_dir, 'column_mapping.json')
    with open(mapping_file, 'w', encoding='utf-8') as f:
        json.dump({
            'column_info': column_info,
            'column_mapping': column_mapping
        }, f, indent=2, ensure_ascii=False)
    print(f"✅ Column mapping saved to {mapping_file}")
    
    # Save as CSV for easy viewing
    csv_file = os.path.join(output_dir, 'wvs_processed.csv')
    df_processed.to_csv(csv_file, index=False, encoding='utf-8')
    print(f"✅ CSV saved to {csv_file}")

# Save results
save_results(df_processed, column_info, column_mapping)

print("\n🎉 ALL DONE! The dataset is now preprocessed and ready for analysis.")
print("Check the 'wvs_processed' folder for output files.")

In [ ]:
# Cell 13: Quick Verification

print("🔍 QUICK VERIFICATION")
print("="*60)

print("Original columns (first 5):")
for col in df.columns[:5]:
    print(f"  {col}")

print("\nProcessed columns (first 5):")
for col in df_processed.columns[:5]:
    print(f"  {col}")

print("\nSample data (first row):")
print(df_processed.iloc[0].to_string())

print("\n✅ Notebook execution complete!")